In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# 1. Configuration constants
N_PASSENGERS = 1000
STATIONS = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L"]

# 2. Define Beta Parameters 
B_DIST = -0.05
B_SHELTER = 0.6
B_RAINY = -0.3
B_DIST_RAINY = -0.02

# Define ASCs 
ASCs = {
    "opt_out": -0.15,
    "A": 0.0,
    "B": 0.1,
    "C": -0.1,
    "D": 0.0,
    "E": 0.2,
    "F": -0.2,
    "G": 0.1,
    "H": 0.0,
    "I": -0.05,
    "J": 0.15,
    "K": -0.15,
    "L": 0.0,
}

# 3. Initialize data list
data_rows = []

# 4. Simulate passenger choices
for passenger_id in range(1, N_PASSENGERS + 1):
    row = {"id": passenger_id, "rainy": np.random.choice([0, 1], p=[0.6, 0.4])}
    rainy_val = row["rainy"]

    utilities = {}

    # Calculate utility for opt-out 
    utilities["opt_out"] = ASCs["opt_out"]

    # Generate station attributes and calculate utilities
    for station in STATIONS:
        dist = round(np.random.uniform(5, 50), 1)
        shelter = np.random.choice([0, 1])

        row[f"distance_{station}"] = dist
        row[f"shelter_{station}"] = shelter

        # Utility formula matching your specification
        utilities[station] = (
            ASCs[station]
            + (B_DIST * dist)
            + (B_SHELTER * shelter)
            + (B_RAINY * rainy_val)
            + (B_DIST_RAINY * dist * rainy_val)
        )

    # Convert utilities to choices using Logit probabilities
    all_alts = ["opt_out"] + STATIONS
    util_values = np.array([utilities[alt] for alt in all_alts])

    # Logit calculation
    exp_utils = np.exp(util_values - np.max(util_values))
    probabilities = exp_utils / np.sum(exp_utils)

    # Draw final choice
    chosen_alt = np.random.choice(all_alts, p=probabilities)
    row["choice"] = chosen_alt

    data_rows.append(row)

# 5. Create DataFrame and export
df_simulated = pd.DataFrame(data_rows)
df_simulated.to_excel("dataset_output.xlsx", index=False)

print("Choice distribution (in %):")
print(df_simulated["choice"].value_counts(normalize=True).round(3) * 100)


Choice distribution (in %):
choice
opt_out    20.0
J           9.1
G           8.7
E           8.4
A           6.8
H           6.6
B           6.5
C           6.2
D           6.0
K           5.9
L           5.7
I           5.2
F           4.9
Name: proportion, dtype: float64
